# **Ingestion and profiling**

In [3]:
import pandas as pd
import os
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RAW_DIR = os.path.join("..","data","raw")
INTERIM_DIR = os.path.join("..","data","interim")
os.makedirs(INTERIM_DIR,exist_ok=True)

## 1.Load with explicit dtypes


###### NOTE on validation_ts it should be read as string, and not parsed with parse_dates. The profiling shows that there are three diffrent timestamp formats miced in this single column - parsing needs to happend deliberately later, with a fallback chain, not silently at load time where an unrecognized format would just become NaT with no record of why.  
###### NOTE on has_shelter and transfer_flag: these are boolean flags but have mixed encoding strings, and they should be normalized to real boolean dtpe later.

In [4]:
def load_csv(name:str, dtype: dict) -> pd.DataFrame:
   path = os.path.join(RAW_DIR, name)
   try:
      return pd.read_csv(path, dtype=dtype)
   except FileNotFoundError:
      raise FileNotFoundError(
         f"Expected file not found: {os.path.abspath(path)}\n"
         f"-> Check that '{name}' has been placed in data/raw/ before running this notebook."
      ) from None

In [5]:
DTYPES = {
    "routes.csv": {
        "route_id": "string", "route_name": "string", "mode": "string",
        "district": "string", "route_leight_mk" : "float64", "schedule_trips_per_day": "Int64"
    },
    "vehicles.csv": {
        "vehicle_id": "string", "vehicle_type": "string", "capacity": "Int64",
         "year_manufactured": "Int64", "depot": "string",
    },
    "stops.csv": {
        "stop_id": "string", "stop_name": "string", "district": "string",
        "zone": "string", "has_schelter" : "string"
    },
    "validations.csv": {
        "validation_id": "string", "validation_ts": "string", "route_id": "string",
        "vehicle_id": "string", "stop_id": "string", "card_id": "string", "fare_type": "string",
        "fare_amount": "float64", "passanger_count": "string", "transfer_flag": "string"
    },
}


In [7]:
frames = {}
for fname, dtypes in DTYPES.items():
    key = fname.replace(".csv", "")
    frames[key] = load_csv(fname, dtypes)

routes, vehicles, stops, validations = (
    frames["routes"],frames["vehicles"],frames["stops"],frames["validations"]
)
print("Loaded:", {k: v.shape for k, v in frames.items()})


Loaded: {'routes': (60, 6), 'vehicles': (180, 5), 'stops': (400, 5), 'validations': (81600, 10)}


## 2. Missing file handling check


###### Missing input files fail with a clear, actionable message rather thana raw traceback —confirmed by deliberately requesting a file that doesn't exist. Theerror names the exact file expected, gives its full path, and tells the reader what to do about it, instead of surfacing pandas' internal stack trace.



In [8]:
try:
    load_csv("does_not_exist.csv", {})
except FileNotFoundError as e:
    print("Caught as expected:")
    print(e)

Caught as expected:
Expected file not found: d:\Zavrsen proekt\data-analysis-bootcamp-capstone-project\data\raw\does_not_exist.csv
-> Check that 'does_not_exist.csv' has been placed in data/raw/ before running this notebook.


## 3.Profile for each file


##### The profile for each file shows us :  
###### ROUTES: district and route_lenght_km each have 3 nulls.route_namme, mode, and scheduled_trips_per_day are complete.  
###### VEHICLES: capacity has 12 nulls (~7%)-the profiling below on numeric ranges also flags this column for outliers, not just missingness. year_manufactured and depot are complete.  
###### STOPS: fully complete, no nulls. The encoding in the has_shelter should be looked at.  
###### VALIDATIONS: card_id is missing on ~12% or rows - plausibly cash/contactless-without-card payments rather than a data error, worth comfirming rather than assuming. fare_amount is missing rows and passanger_count is missing rows.


In [9]:
def profile(df: pd.DataFrame, name: str) -> pd.DataFrame:
    print(f"{'=' * 70}\nPROFILE: {name}\n{'=' * 70}")
    print(f"rows: {len(df):,}   columns: {df.shape[1]}")
    print(f"memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    null_counts = df.isna().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": null_counts,
        "null_pct": null_pct,
    })
    display(summary)
    display(df.head())
    return summary


route_summary = profile(routes, "routes")

PROFILE: routes
rows: 60   columns: 6
memory usage: 0.01 MB


,dtype,null_count,null_pct
route_id,string,0,0.0
route_name,string,0,0.0
mode,string,0,0.0
district,string,3,5.0
route_length_km,float64,3,5.0
scheduled_trips_per_day,int64,0,0.0


,route_id,route_name,mode,district,route_length_km,scheduled_trips_per_day
0,R001,Angel Harbor Line,tram,Harbourside,19.8,133
1,R002,Jeffrey Street Line,BUS,<NA>,12.6,80
2,R003,Erica Plains Line,METRO,Southbank,4.3,203
3,R004,Brittany Bypass Line,Tram,Northgate,17.6,131
4,R005,Clayton Fort Line,bus,<NA>,5.0,159


In [10]:
vehicles_summary = profile(vehicles, "vehicles")

PROFILE: vehicles
rows: 180   columns: 5
memory usage: 0.03 MB


,dtype,null_count,null_pct
vehicle_id,string,0,0.00
vehicle_type,string,0,0.00
capacity,Int64,12,6.67
year_manufactured,Int64,0,0.00
depot,string,0,0.00


,vehicle_id,vehicle_type,capacity,year_manufactured,depot
0,V0001,Articulated Bus,125,2021,Central Depot
1,V0002,Standard Bus,<NA>,1998,East Depot
2,V0003,Articulated Bus,118,2022,Central Depot
3,V0004,Tram,170,2008,Central Depot
4,V0005,Standard Bus,73,2013,Riverside Depot


In [11]:
validations_summary = profile(validations, "validations")

PROFILE: validations
rows: 81,600   columns: 10
memory usage: 36.62 MB


,dtype,null_count,null_pct
validation_id,string,0,0.00
validation_ts,string,0,0.00
route_id,string,0,0.00
vehicle_id,string,0,0.00
stop_id,string,0,0.00
card_id,string,9777,11.98
fare_type,string,0,0.00
fare_amount,float64,1183,1.45
passenger_count,float64,833,1.02
transfer_flag,string,0,0.00


,validation_id,validation_ts,route_id,vehicle_id,stop_id,card_id,fare_type,fare_amount,passenger_count,transfer_flag
0,VAL-0068918,2024-05-13 18:54:29,R001,V0044,S0234,CARD-001739,Full Fare,2.53,1.0,N
1,VAL-0044034,06/06/2024 08:48,R058,V0086,S0131,CARD-000729,ADULT,2.48,1.0,Y
2,VAL-0017450,06/03/2024 13:15,R043,V0177,S0251,CARD-020205,Adult,2.85,1.0,FALSE
3,VAL-0004736,2024-04-21 17:30:00,R026,V0108,S0330,CARD-013092,student,1.64,3.0,Y
4,VAL-0053735,2024-06-14 18:35:27,R003,V0057,S0389,CARD-003251,Student,1.34,1.0,0


In [12]:
stops_summary = profile(stops, "stops")

PROFILE: stops
rows: 400   columns: 5
memory usage: 0.11 MB


,dtype,null_count,null_pct
stop_id,string,0,0.0
stop_name,string,0,0.0
district,string,0,0.0
zone,string,0,0.0
has_shelter,str,0,0.0


,stop_id,stop_name,district,zone,has_shelter
0,S0001,matthew radial stop,Riverside,Zone 2,N
1,S0002,Brianna Avenue Stop,Centro,Zone 3,1
2,S0003,jean circle stop,Northgate,Zone 2,0
3,S0004,Carlos Bypass Stop,Old Town,Zone 1,Y
4,S0005,JONES FREEWAY STOP,Riverside,Zone 2,N


# 4. Categorical columns(value_counts())

#### Notes:

###### - **`routes.mode`**: the same three categories (bus/tram/metro) show up under multiple spellings — lowercase, uppercase, capitalized — and 12 rows carry stray leading/trailing whitespace (`"BUS "`, etc.) on top of the casing issue.
###### - **`stops.has_shelter`**: a boolean flag encoded three different ways at once — `Y`/`N`, `1`/`0`. No nulls, but four distinct string values need to collapse to two boolean states.
###### - **`validations.transfer_flag`**: same boolean-flag problem, worse — six distinct values (`Y`/`N`, `0`/`1`, `TRUE`/`FALSE`). A simple `.str.lower()` won't fix this; needs an explicit value-mapping table.
###### - **`validations.fare_type`**: this is the big one. It's not just casing — there appear to be **17 distinct labels for what looks like 4 real fare categories**: `Full Fare`/`ADULT`/`Adult`/`adult` are plausibly all "adult fare"; `Kid`/`child`/`Child` are plausibly all "child fare"; `Senior`/`Senior Citizen`/`senior`; `Pass Holder`/`PassHolder`/`Monthly Pass`/`pass holder`. Whether `Full Fare`truly means the same thing as `Adult`, or is a genuinely distinct fare product, is a judgment call notebook 02 needs to make explicitly (and document) — this can't be resolved by mechanical case-folding alone.
###### - **`vehicles.vehicle_type`** and **`vehicles.depot`**: look clean — consistent casing, no stray whitespace, worth re-confirming after the other fixes land but not currently flagged.


In [14]:
CATEGORICAL_COLUMNS = {
    "routes": ["mode","district"],
    "vehicles": ["vehicles_type", "depot"],
    "stops": ["district", "zone", "transfer_flag"],
    
}

# 5.Numeric range check


#### Notes

###### - **`vehicles.year_manufactured`**: contains values of `1899` and `2035` — one impossibly old (predates motor vehicles), one in the future relative to this dataset. These read as data-entry sentinel/placeholder values rather than real manufacture years and need an explicit rule in notebook 02 (most likely: quarantine, not guess a replacement).
###### - **`vehicles.capacity`**: a small number of vehicles (~19) sit well above the IQR-based outlier threshold, up to 690 — worth checking whether these are legitimately large metro consists or data errors, ideally against `vehicle_type` (a "Metro Set" at 690 is plausible; a "Standard Bus" at 690 is not).
###### - **`validations.passenger_count`**: includes negative values (`-1`, `-2`, `-3`) alongside normal positive counts. A negative passenger count is impossible — this looks like a sign got flipped somewhere upstream, and `abs()` is a reasonable candidate fix, but that's a notebook 02 decision to make and document, not something to silently correct here.
###### - **`validations.fare_amount`**: the vast majority of values sit in a tight, sensible range (~1–3), but 25 rows are round, implausibly large numbers — `9999.00` (7 rows), `5000.00` (7 rows), `2000.00` (6 rows), `1234.56` (5 rows). These look like sentinel/placeholder values rather than real fares (real fares aren't round to the dollar, let alone `1234.56`), and they inflate the mean and std dramatically (mean 3.37 vs. median 2.31). Notebook 02 should treat all of these as missing, not as legitimate high fares — checking for exact matches against this small set of round numbers, not just an arbitrary threshold.

In [16]:
print("--- vehicles.year_manufactured range ---")
print(vehicles["year_manufactured"].describe())
suspicious_years = vehicles[(vehicles["year_manufactured"] < 1950) | (vehicles["year_manufactured"] > 2025)]
print(f"\nvehicles with implausible year_manufactured: {len(suspicious_years)}")
display(suspicious_years[["vehicle_id", "vehicle_type", "year_manufactured"]])

print("\n---vehicles.capacity range --- ")
print(vehicles["capacity"].describe())
q1, q3 = vehicles["capacity"].quantile([0.25, 0.75])
iqr = q3 - q1
cap_outliers = vehicles[vehicles["capacity"]> q3 + 1.5 * iqr]
print(f"capacity outliers (> Q3 + 1.5 * IQR): {len(cap_outliers)}")

print("\n--- validations.passanger_count range ---")
print(validations["passenger_count"].value_counts(dropna=False).sort_index())

print("\n--- validations.fare_amount range ---")
print(validations["fare_amount"].describe())
fare_outliers = validations[validations["fare_amount"] > 100]
print(f"\nfare_amount rows > 100: {len(fare_outliers)}")
print(fare_outliers["fare_amount"].value_counts())


--- vehicles.year_manufactured range ---
count          180.0
mean     2003.911111
std        20.613303
min           1899.0
25%           1994.0
50%           2006.0
75%           2016.0
max           2035.0
Name: year_manufactured, dtype: Float64

vehicles with implausible year_manufactured: 15


,vehicle_id,vehicle_type,year_manufactured
54,V0055,Standard Bus,1899
56,V0057,Metro Set,1899
65,V0066,Standard Bus,2035
76,V0077,Articulated Bus,2035
82,V0083,Standard Bus,2035
89,V0090,Standard Bus,1899
91,V0092,Standard Bus,2035
119,V0120,Standard Bus,2035
132,V0133,Standard Bus,2035
142,V0143,Standard Bus,2035



---vehicles.capacity range --- 
count         168.0
mean     169.416667
std      165.353492
min            60.0
25%            70.0
50%           111.5
75%          193.25
max           690.0
Name: capacity, dtype: Float64
capacity outliers (> Q3 + 1.5 * IQR): 19

--- validations.passanger_count range ---
passenger_count
-3.0      126
-2.0      106
-1.0      130
 1.0    75549
 2.0     2400
 3.0     2456
 NaN      833
Name: count, dtype: int64

--- validations.fare_amount range ---
count    80417.000000
mean         3.367416
std        106.137290
min          0.000000
25%          1.230000
50%          2.310000
75%          2.520000
max       9999.000000
Name: fare_amount, dtype: float64

fare_amount rows > 100: 25
fare_amount
9999.00    7
5000.00    7
2000.00    6
1234.56    5
Name: count, dtype: int64


In [19]:
for key, df in frames.items():
    out_path = os.path.join( INTERIM_DIR, f"{key}.pkl")
    df.to_pickle(out_path)
    print(f"saved{out_path} ({len(df):,} rows)")


saved..\data\interim\routes.pkl (60 rows)
saved..\data\interim\vehicles.pkl (180 rows)
saved..\data\interim\stops.pkl (400 rows)
saved..\data\interim\validations.pkl (81,600 rows)
